# 12 — The forecast package for Part 3

**Does:** collects everything Part 2 knows about tomorrow into one table per day and area: demand (with a 10–90% range), available supply by fuel, price (with a range), tightness, and whether tomorrow sits near a coal-tier step. Each forecast column uses only what was known the day before; the matching actual columns are alongside for evaluation.

**Why:** in Part 3, large generators bid strategically, and their view of tomorrow is this package. Market power should bite on forecast-tight days and near steps.

**Input:** Stage 3–5 outputs (`demand_forecast_dayahead.csv`, `availability_forecast_test.csv`, `price_forecast_test.csv`), plus the Stage 3 features for the demand ranges

**Output:** `data/processed/part3_forecast_package.csv`, read with `load_forecast_package()` in `src/forecast.py`

**Logged in:** `docs/model_log.md`, Step 20

| Column group | Source | Known on day D? |
|---|---|---|
| Demand forecast, Q10, Q90 | L6 LightGBM median / 10% / 90%, trained on actual weather, fed anchored real day-ahead weather (Stages 3–4) | ✅ |
| Available MW by fuel | "best day-ahead" availability for solar, wind, hydro; constant factors for coal and others (Stage 4) | ✅ |
| Price forecast, low, high | Route B: forecast demand and availability through dispatch; low/high from the Q10/Q90 demand (Stage 5) | ✅ |
| Tightness forecast | (forecast available − forecast demand) / forecast demand | ✅ |
| Near step (forecast) | Dispatch with forecast demand +2%: national price rises ≥ 100 Rs/MWh | ✅ |
| *_actual | What happened (the actual-price run of Step 15) | ❌ outcomes |

**National rows:** demand and supply are sums; prices are demand-weighted. The national demand Q10/Q90 are **sums of state quantiles**, which is wider than a true national 10–90% range (state errors partly cancel); kept as a conservative band.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

import weather as w
import features as F
import forecast as fc
from dispatch import (LossDispatchModel, daily_availability_lookup, load_coordinates, load_demand,
                      load_generators, loss_matrix, run_all_days, PROCESSED)

SCORE = slice("2024-03-01", "2025-03-31")
dates = pd.date_range(SCORE.start, SCORE.stop)
gen = load_generators()
demand = load_demand()
states = list(demand.columns)
model = LossDispatchModel(gen, loss_matrix(load_coordinates(), states), states)

## 1. Demand: forecast and 10–90% range

In [2]:
dfc = pd.read_csv(PROCESSED / "demand_forecast_dayahead.csv", parse_dates=["Date"])
dem_fc = dfc[(dfc.Model == "L6 LightGBM median") & (dfc.Weather == "day-ahead")].pivot(
    index="Date", columns="State", values="Forecast").reindex(dates)[states]

sw = pd.read_csv(PROCESSED / "weather_state_daily.csv", parse_dates=["Date"])
fcw = pd.read_csv(PROCESSED / "forecast_weather_state_daily.csv", parse_dates=["Date"])
hol, pad = F.load_holidays(), F.load_paddy()
f_perfect = F.build_features(demand, sw, hol, pad, "perfect")
f_dayahead = F.build_features(demand, sw, hol, pad, "day-ahead", forecast_weather=w.anchor_forecast(sw, fcw))
q = {}
for alpha in (0.1, 0.9):
    p = fc.walk_forward(f_perfect, lambda: fc.Trees(fc.FEATURE_SETS["L4"], quantile=alpha),
                        start=SCORE.start, test_f=f_dayahead)
    q[alpha] = p.pivot(index="Date", columns="State", values="Forecast").reindex(dates)[states]
q10, q90 = np.minimum(q[0.1], q[0.9]), np.maximum(q[0.1], q[0.9])     # guard against crossed quantiles
dem_act = demand.loc[SCORE]
inside = ((dem_act >= q10) & (dem_act <= q90)).stack().mean()
print(f"actual state demand inside the 10–90% range: {inside:.1%}")
assert dem_fc.notna().all().all() and q10.notna().all().all()

actual state demand inside the 10–90% range: 74.6%


## 2. Supply: available MW by fuel

In [3]:
afc = pd.read_csv(PROCESSED / "availability_forecast_test.csv", parse_dates=["Date"])
best = afc[afc.Model == "best day-ahead"][["Date", "State", "Fuel_Type", "Forecast"]].rename(columns={"Forecast": "Factor"})
act_avail = pd.read_csv(PROCESSED / "availability_daily.csv", parse_dates=["Date"])

GROUP = {"Solar": "Solar", "Wind": "Wind", "Large Hydro": "Hydro", "Small Hydro": "Hydro", "Coal": "Coal"}
def by_group(daily):
    mw = fc.available_by_fuel(gen, daily[daily.Date.between(SCORE.start, SCORE.stop)], dates)
    mw["Group"] = mw.Fuel_Type.map(GROUP).fillna("Other")
    return mw.pivot_table(index=["Date", "State"], columns="Group", values="Available_MW", aggfunc="sum", fill_value=0)   # 0 MW where a state has none

supply_fc, supply_act = by_group(best), by_group(act_avail)
print("India average available MW by group (forecast):")
print((supply_fc.groupby(level="Date").sum().mean() / 1000).round(1).rename("GW").to_string())

India average available MW by group (forecast):
Group
Coal     162.2
Hydro     24.5
Other     24.0
Solar     41.4
Wind      16.4


## 3. Price, range and tightness (Stage 5) and the step-edge flag

In [4]:
pf = pd.read_csv(PROCESSED / "price_forecast_test.csv", parse_dates=["Date"])
real = pf[(pf.Route == "B") & (pf.Run == "realistic")].set_index(["Date", "Area"])

lookup_fc = daily_availability_lookup(best[best.Date.between(SCORE.start, SCORE.stop)])
_, _, nat_fc = run_all_days(model, dem_fc, daily=lookup_fc)
_, _, nat_up = run_all_days(model, dem_fc * 1.02, daily=lookup_fc)
nat_fc, nat_up = nat_fc.set_index("Date").Price_avg, nat_up.set_index("Date").Price_avg
assert (nat_fc - real.xs("National", level="Area").Forecast).abs().max() < 0.01, "must match Stage 5's realistic run (saved to 4 decimals)"
near_fc = (nat_up - nat_fc) >= 100

lookup_act = daily_availability_lookup(act_avail[act_avail.Date.between(SCORE.start, SCORE.stop)])
_, _, act_up = run_all_days(model, dem_act * 1.02, daily=lookup_act)
actual_nat = pd.read_csv(PROCESSED / "dispatch_weather_national_daily.csv", parse_dates=["Date"]).set_index("Date").Price_avg.loc[SCORE]
near_act = (act_up.set_index("Date").Price_avg - actual_nat) >= 100

agree = (near_fc == near_act).mean()
caught = (near_fc & near_act).sum() / near_act.sum()
print(f"near-step days: forecast {near_fc.sum()}, actual {near_act.sum()}; "
      f"forecast flags {caught:.0%} of the actual near-step days; same answer on {agree:.0%} of days")

near-step days: forecast 82, actual 82; forecast flags 50% of the actual near-step days; same answer on 79% of days


## 4. Assemble

In [5]:
def state_table():
    t = pd.DataFrame({
        "Demand_forecast": dem_fc.stack(), "Demand_Q10": q10.stack(), "Demand_Q90": q90.stack(),
        "Demand_actual": dem_act.stack()})
    t.index.names = ["Date", "Area"]
    s_fc = supply_fc.rename_axis(["Date", "Area"]).add_suffix("_MW_forecast")
    t = t.join(s_fc).join(supply_act.sum(axis=1).rename("Available_MW_actual").rename_axis(["Date", "Area"]))
    t["Available_MW_forecast"] = supply_fc.sum(axis=1).rename_axis(["Date", "Area"])
    return t

st = state_table()
nat = st.groupby(level="Date").sum()                                   # sums for India
nat.index = pd.MultiIndex.from_product([nat.index, ["National"]], names=["Date", "Area"])
pkg = pd.concat([st, nat]).sort_index()
pkg = pkg.join(real[["Forecast", "Low", "High", "Actual", "Tightness_forecast", "Tightness_actual"]].rename(
    columns={"Forecast": "Price_forecast", "Low": "Price_low", "High": "Price_high", "Actual": "Price_actual"}))
pkg = pkg.join(pd.DataFrame({"Price_up2pct_forecast": nat_up, "Near_step_forecast": near_fc,
                             "Near_step_actual": near_act}), on="Date")
pkg = pkg.reset_index()[fc.PACKAGE_COLUMNS]

# checks
assert len(pkg) == len(dates) * 33 and pkg.drop(columns=["Price_low", "Price_high"]).notna().all().all()
assert (pkg.Price_low <= pkg.Price_high).all() and (pkg.Demand_Q10 <= pkg.Demand_Q90).all()
n = pkg[pkg.Area == "National"].set_index("Date")
assert np.allclose(n.Demand_forecast, dem_fc.sum(axis=1)) and np.allclose(n.Demand_actual, dem_act.sum(axis=1))
pkg.to_csv(PROCESSED / "part3_forecast_package.csv", index=False, float_format="%.2f")
pkg2 = fc.load_forecast_package()
print(f"saved {len(pkg):,} rows ({len(dates)} days × 33 areas); loader OK: {list(pkg2.columns) == fc.PACKAGE_COLUMNS}")
pkg[pkg.Area == "National"].describe().T[["mean", "min", "max"]].round(2)

saved 13,068 rows (396 days × 33 areas); loader OK: True


,mean,min,max
Date,2024-09-14 12:00:00,2024-03-01 00:00:00,2025-03-31 00:00:00
Demand_forecast,222075.302525,174208.1,253364.0
Demand_Q10,213185.347092,165691.940491,245885.169368
Demand_Q90,230126.216663,182831.885407,261663.916775
Demand_actual,221298.752525,177363.0,251742.0
Solar_MW_forecast,41436.100994,27654.555282,52650.507074
Wind_MW_forecast,16448.715446,795.700356,56133.45624
Hydro_MW_forecast,24526.190729,10912.339948,45649.142451
Coal_MW_forecast,162245.6325,162245.6325,162245.6325
Other_MW_forecast,23981.135,23981.135,23981.135


## 5. A first look for Part 3

Tight days by forecast: how often does tomorrow look tight, and what happens to the price then?

In [6]:
n = pkg[pkg.Area == "National"].set_index("Date")
groups = pd.cut(n.Tightness_forecast, [-1, 0.10, 0.15, 0.20, 0.30, 1],
                labels=["< 10% spare", "10–15%", "15–20%", "20–30%", "> 30%"])
pd.DataFrame({"days": groups.value_counts().sort_index(),
              "actual price": n.Price_actual.groupby(groups, observed=True).mean(),
              "forecast price": n.Price_forecast.groupby(groups, observed=True).mean(),
              "near a step (forecast)": n.Near_step_forecast.groupby(groups, observed=True).mean()}).round(2)

,days,actual price,forecast price,near a step (forecast)
Tightness_forecast,,,,
< 10% spare,44,3735.58,3754.45,0.11
10–15%,93,3588.29,3619.71,0.41
15–20%,67,3231.38,3298.70,0.43
20–30%,112,2802.83,2804.44,0.04
> 30%,80,2654.64,2682.45,0.06


CONCLUSIONS_PLACEHOLDER